<a href="https://colab.research.google.com/github/bmulaw/ComputerVision/blob/main/ComputerVision_lab3_openWorldVisionipynb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Computer Vision Lecture 2 Practical Session 2
### *Open-World Vision*
## Team Name/Member: Bernard MULAW
#### Date: 23/11/2026

In [1]:
import platform
print(platform.python_version())

3.13.18


In [2]:
%pip -q install "torchvision" "groundingdino-py" "segment-anything" "transformers>=4.35.3" "bitsandbytes"

In [3]:
import os
import cv2
import torch
import numpy as np

from PIL import Image
from matplotlib import pyplot as plt
from matplotlib import image as mpimg
from google.colab import drive, files

from groundingdino.util.inference import load_model
from segment_anything import SamPredictor, sam_model_registry, SamAutomaticMaskGenerator
from transformers import LlavaForConditionalGeneration

/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [4]:
print(torch.__version__)
print("cuda is available =", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"

2.14.1+cpu
cuda is available = False


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Task 0.1: Verify Model Loading

In [6]:
# Got from here: https://pypi.org/project/groundingdino-py/
!git clone https://github.com/IDEA-Research/GroundingDINO.git
!cd GroundingDINO/
%pip install -q -e .
%mkdir weights
%cd weights
!wget -q "https://github.com/IDEA-Research/GroundingDINO/releases/download/v0.1.0-alpha/groundingdino_swint_ogc.pth"

fatal: destination path 'GroundingDINO' already exists and is not an empty directory.
ERROR: file:///content does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
mkdir: cannot create directory ‘weights’: File exists
/content/weights


In [7]:

%ls weights/ GroundingDINO/groundingdino/config
# %cd ../


ls: cannot access 'GroundingDINO/groundingdino/config': No such file or directory
weights/:
GroundingDINO/  groundingdino_swint_ogc.pth  weights/
/content


In [8]:
%ls weights/

groundingdino_swint_ogc.pth    groundingdino_swint_ogc.pth.2  weights/
groundingdino_swint_ogc.pth.1  groundingdino_swint_ogc.pth.3


In [9]:
# Load Grounding DINO, SAM, and LLaVA (using 4-bit to save moemory) models
dino = load_model("GroundingDINO/groundingdino/config/GroundingDINO_SwinT_OGC.py",
                   "weights/groundingdino_swint_ogc.pth")

sam = sam_model_registry["vit_h"](checkpoint="sam_vit_h_4b8939.pth").to(device=device)

llava = LlavaForConditionalGeneration.from_pretrained(
    "llava -hf/llava -1.5 -7b-hf",
    torch_dtype = torch.float16,
    load_in_4bit = True).to(0)

print("All models loaded successfully")

/usr/local/lib/python3.13/dist-packages/torch/functional.py:504: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/TensorShape.cpp:4221.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]


final text_encoder_type: bert-base-uncased


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


AttributeError: 'BertModel' object has no attribute 'get_extended_attention_mask'